In [0]:
def read_csv_df(spark,path,header=True,infer_schema=True,sep=","):
    return (spark.read
        .option("header", header)
        .option("inferSchema", infer_schema)
        .option("sep", sep)
        .csv(path))

df1=read_csv_df(spark,'/Volumes/logistics_catalog_assign/landing_zone/landing_vol/logistics_source1.txt',True,True,',')
df1.display()
df2=read_csv_df(spark,'/Volumes/logistics_catalog_assign/landing_zone/landing_vol/logistics_source2.txt',True,True,',')
df2.display()

In [0]:
#UDF1: Complex Incentive Calculation

from pyspark.sql.functions import udf,StringType,col

def bonus_cal(age,role):
    if role == 'Driver' and age > 50:
        Bonus = '15%'
    elif role == 'Driver' and age < 50:
        Bonus = '5%'
    else:
        Bonus = '0'
    return Bonus

bonus_cal_udf = udf(bonus_cal,StringType())

df1 = df1.withColumn('projected_bonus',bonus_cal_udf(col('age'),col('role')))
df1.display()
df2 = df2.withColumn('projected_bonus',bonus_cal_udf(col('age'),col('role')))
df2.display()


In [0]:
#UDF2: PII Masking (Privacy Compliance)
from pyspark.sql.functions import col,concat,lit,mask,substring,repeat,col,length,expr

def datamask(df, colname):
    df = df.withColumn(
        colname,
        concat(
            substring(col(colname), 1, 2),  # first 2 characters
            expr(f"repeat('*', length({colname}) - 3)"),
            substring(col(colname), -1, 1)))  # last character
    
    return df
    
df1 = datamask(df1,'first_name')
df1 = datamask(df1,'last_name')
df1.display()
